# 04. Macro F1 개선 실험

**목표:** 현재 기준선(LightGBM, macro F1 ≈ 0.824)을 0.84 수준까지 끌어올린다.

**실험 설계 원칙 (발표 방어용)**
1. 모든 실험은 **완전히 동일한 데이터 분할**(5-fold × 3회 반복 = 15개 폴드)을 재사용한다.
   → 성능 차이가 분할 운 때문인지, 방법 때문인지 구분할 수 있다.
2. 전처리·리샘플링·구간화는 **학습 폴드에서만** 수행한다. (데이터 누수 방지)
3. 판정 기준 조정(실험 C)은 **중첩 교차검증**으로 평가한다.
   → 평가에 쓸 데이터로 기준을 맞추면 점수가 부풀려지므로.
4. 기준선과의 비교는 폴드별 점수를 **짝지어** Wilcoxon 부호순위 검정으로 확인한다.

> 셀을 위에서부터 순서대로 실행하면 된다. 실험 A~E는 독립적이라 원하는 것만 골라 돌려도 된다.


In [ ]:
# ── 0. 설정 및 라이브러리 ──────────────────────────────────────────
# 필요 시 설치:  uv add lightgbm xgboost optuna imbalanced-learn scikit-learn pandas matplotlib

import warnings, time, pickle
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from scipy.stats import wilcoxon

from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

SEED      = 42
DATA_PATH = "data/raw/Faults_processed.csv"   # ← 본인 경로에 맞게 수정
N_SPLITS, N_REPEATS = 5, 3

plt.rcParams["font.family"] = "Malgun Gothic"   # Windows. mac은 "AppleGothic"
plt.rcParams["axes.unicode_minus"] = False
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)
print("준비 완료")

In [ ]:
# ── 1. 데이터 로드 & 공통 실험 하네스 ─────────────────────────────
df = pd.read_csv(DATA_PATH)
y_raw = df.pop("target")

le      = LabelEncoder().fit(y_raw)
y       = le.transform(y_raw)
CLASSES = list(le.classes_)
X       = df.copy()

# 모든 실험이 공유할 고정 폴드
_cv   = RepeatedStratifiedKFold(n_splits=N_SPLITS, n_repeats=N_REPEATS, random_state=SEED)
FOLDS = list(_cv.split(X, y))

print(f"데이터 {X.shape[0]}행 × {X.shape[1]}변수 / 클래스 {len(CLASSES)}개 / 폴드 {len(FOLDS)}개")
print(pd.Series(y_raw).value_counts().to_frame("개수"))


def evaluate(make_model, X=X, y=y, folds=FOLDS, weights=None, return_oof=False, tag=""):
    '''고정 폴드에서 모델을 학습·평가하고 폴드별 macro F1 배열을 반환.

    make_model : 인자 없이 호출하면 '학습되지 않은 새 모델'을 주는 함수
    weights    : 클래스별 확률 보정 가중치 (실험 C에서 사용). None이면 일반 argmax.
    '''
    scores, oof_list = [], []
    t0 = time.time()
    for i, (tr, te) in enumerate(folds):
        m = make_model()
        m.fit(X.iloc[tr], y[tr])
        proba = m.predict_proba(X.iloc[te])
        pred  = (proba * weights).argmax(1) if weights is not None else proba.argmax(1)
        scores.append(f1_score(y[te], pred, average="macro"))
        if return_oof:
            oof_list.append((te, proba))
    scores = np.array(scores)
    print(f"{tag:34s} macro F1 = {scores.mean():.4f} ± {scores.std():.4f}   ({time.time()-t0:.0f}초)")
    return (scores, oof_list) if return_oof else scores


def compare(scores, baseline, tag=""):
    '''기준선과 짝지은 비교 + Wilcoxon 부호순위 검정'''
    diff = scores - baseline
    p = wilcoxon(scores, baseline)[1] if np.any(diff != 0) else 1.0
    print(f"{tag:34s} 평균차 {diff.mean():+.4f}   15폴드 중 {int((diff>0).sum())}회 우세   p={p:.4f}")
    return dict(실험=tag, macro_f1=scores.mean(), 표준편차=scores.std(),
                기준대비=diff.mean(), 우세폴드=int((diff > 0).sum()), p값=p)

RESULTS = []

## 기준선 재현

이후 모든 실험은 이 숫자와 비교한다. 먼저 현재 상태를 다시 찍어 기록한다.

In [ ]:
# ── 2. 기준선 ────────────────────────────────────────────────────
def lgb_base():
    return LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=31,
                          subsample=0.8, colsample_bytree=0.8,
                          random_state=SEED, n_jobs=-1, verbose=-1)

base_scores = evaluate(lgb_base, tag="① 기준선 (LightGBM 기본)")
RESULTS.append(dict(실험="① 기준선", macro_f1=base_scores.mean(), 표준편차=base_scores.std(),
                    기준대비=0.0, 우세폴드=0, p값=np.nan))

## 실험 A. 클래스 가중치

**아이디어.** 지금 모델은 1941개 샘플을 모두 똑같은 무게로 본다. 그러면 673개인 Other_Faults를
맞히는 게 55개인 Dirtiness를 맞히는 것보다 12배 이득이라서, 모델은 자연스럽게 희귀 결함을 포기한다.
그런데 macro F1은 **클래스마다 점수를 따로 내서 단순 평균**하므로, Dirtiness 하나를 포기하는 순간
전체 점수의 1/7을 잃는다. 학습 목표와 평가 지표가 어긋나 있는 상태다.

`class_weight="balanced"`는 **희귀 클래스 샘플 하나를 더 비싸게** 취급해 이 어긋남을 줄인다.

In [ ]:
# ── 3. 실험 A: 클래스 가중치 ─────────────────────────────────────
def lgb_balanced():
    return LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=31,
                          subsample=0.8, colsample_bytree=0.8, class_weight="balanced",
                          random_state=SEED, n_jobs=-1, verbose=-1)

s = evaluate(lgb_balanced, tag="② 클래스 가중치 balanced")
RESULTS.append(compare(s, base_scores, "② 클래스 가중치 balanced"))

# 가중치 강도를 직접 조절해보기 (balanced가 과하면 제곱근으로 완화)
counts = np.bincount(y)
soft_w = {k: (counts.sum() / (len(counts) * c)) ** 0.5 for k, c in enumerate(counts)}

def lgb_softw():
    return LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=31,
                          subsample=0.8, colsample_bytree=0.8, class_weight=soft_w,
                          random_state=SEED, n_jobs=-1, verbose=-1)

s = evaluate(lgb_softw, tag="③ 클래스 가중치 완화(제곱근)")
RESULTS.append(compare(s, base_scores, "③ 클래스 가중치 완화(제곱근)"))

## 실험 B. 하이퍼파라미터 튜닝

**아이디어.** 지금 설정값은 내가 임의로 정한 것이다. 탐색 목표를 정확도가 아니라
**macro F1 자체**로 두고 찾으면, 소수 클래스를 살리는 방향의 설정이 선택된다.

탐색은 **3-fold × 1회**(빠른 평가)로 돌리고, 찾은 설정은 **15폴드 전체**에서 다시 검증한다.
탐색에 쓴 분할로 최종 점수를 보고하면 그 점수는 부풀려져 있다.

In [ ]:
# ── 4. 실험 B: Optuna 하이퍼파라미터 탐색 ────────────────────────
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

tune_folds = list(StratifiedKFold(3, shuffle=True, random_state=777).split(X, y))

def objective(trial):
    p = dict(
        n_estimators     = trial.suggest_int("n_estimators", 200, 1200, step=100),
        learning_rate    = trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
        num_leaves       = trial.suggest_int("num_leaves", 15, 127, log=True),
        min_child_samples= trial.suggest_int("min_child_samples", 3, 40),
        subsample        = trial.suggest_float("subsample", 0.6, 1.0),
        subsample_freq   = 1,
        colsample_bytree = trial.suggest_float("colsample_bytree", 0.5, 1.0),
        reg_alpha        = trial.suggest_float("reg_alpha", 1e-3, 5.0, log=True),
        reg_lambda       = trial.suggest_float("reg_lambda", 1e-3, 5.0, log=True),
        class_weight     = trial.suggest_categorical("class_weight", [None, "balanced"]),
    )
    sc = []
    for tr, te in tune_folds:
        m = LGBMClassifier(**p, random_state=SEED, n_jobs=-1, verbose=-1)
        m.fit(X.iloc[tr], y[tr])
        sc.append(f1_score(y[te], m.predict(X.iloc[te]), average="macro"))
    return np.mean(sc)

study = optuna.create_study(direction="maximize",
                            sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(objective, n_trials=60, show_progress_bar=True)   # 시간 되면 150~300으로
BEST = study.best_params
print("\n탐색용 분할 점수:", round(study.best_value, 4))
print(pd.Series(BEST).to_frame("최적값").to_string())

In [ ]:
# 찾은 설정을 '전체 15폴드'에서 정식 검증
def lgb_tuned():
    return LGBMClassifier(**BEST, random_state=SEED, n_jobs=-1, verbose=-1)

tuned_scores = evaluate(lgb_tuned, tag="④ 하이퍼파라미터 튜닝")
RESULTS.append(compare(tuned_scores, base_scores, "④ 하이퍼파라미터 튜닝"))

## 실험 C. 클래스별 판정 기준 조정 ★ 핵심

**아이디어.** 모델은 7개 클래스의 확률을 내놓고, 우리는 그중 **가장 큰 것**을 고른다.
이 규칙은 "많이 맞히기"(정확도)에는 최적이지만 macro F1에는 최적이 아니다.

희귀 클래스는 확률이 애초에 작게 나오도록 학습되기 때문에, 실제로 Dirtiness인 샘플에서도
Dirtiness 확률이 2등·3등에 머물러 한 번도 선택받지 못하는 일이 생긴다.
그래서 클래스마다 확률에 **배율(가중치)** 을 곱해 선택 문턱을 다르게 준다.
희귀 클래스는 배율을 키워 조금만 의심스러워도 선택되게 하는 것이다.

**중요 — 평가 방식.** 이 배율을 평가 데이터에서 맞추면 당연히 점수가 오른다(부정행위).
그래서 **중첩 교차검증**을 쓴다. 바깥 폴드의 학습 데이터 안에서 다시 내부 교차검증을 돌려
배율을 찾고, 그 배율을 **한 번도 보지 않은** 바깥 평가 데이터에 적용한다.
발표에서 "그 가중치는 어디서 구했나요?"라는 질문이 반드시 나오므로, 이 구조를 설명할 수 있어야 한다.

In [ ]:
# ── 5. 실험 C: 클래스별 확률 배율 (중첩 교차검증) ────────────────
def tune_weights(proba, y_true, n_round=10, grid=np.linspace(0.4, 4.0, 37)):
    '''좌표상승법: 한 클래스씩 배율을 바꿔보며 macro F1이 좋아지면 채택'''
    w    = np.ones(proba.shape[1])
    best = f1_score(y_true, (proba * w).argmax(1), average="macro")
    for _ in range(n_round):
        improved = False
        for k in range(len(w)):
            cur = w[k]
            for g in grid:
                w[k] = g
                sc = f1_score(y_true, (proba * w).argmax(1), average="macro")
                if sc > best + 1e-6:
                    best, cur, improved = sc, g, True
            w[k] = cur
        if not improved:
            break
    return w, best


def evaluate_with_weight_tuning(make_model, n_inner=4, tag=""):
    '''바깥 폴드마다: 학습셋 내부에서 배율을 학습 → 바깥 평가셋에 적용'''
    scores, w_log = [], []
    t0 = time.time()
    for oi, (tr, te) in enumerate(FOLDS):
        Xtr, ytr = X.iloc[tr], y[tr]

        # (1) 내부 교차검증으로 학습셋의 OOF 확률 생성
        inner = StratifiedKFold(n_inner, shuffle=True, random_state=SEED + oi)
        oof = np.zeros((len(tr), len(CLASSES)))
        for itr, ite in inner.split(Xtr, ytr):
            mi = make_model()
            mi.fit(Xtr.iloc[itr], ytr[itr])
            oof[ite] = mi.predict_proba(Xtr.iloc[ite])

        # (2) OOF 확률 위에서만 배율 탐색
        w, _ = tune_weights(oof, ytr)
        w_log.append(w)

        # (3) 전체 학습셋으로 재학습 → 바깥 평가셋에 배율 적용
        m = make_model()
        m.fit(Xtr, ytr)
        pred = (m.predict_proba(X.iloc[te]) * w).argmax(1)
        scores.append(f1_score(y[te], pred, average="macro"))

    scores = np.array(scores)
    print(f"{tag:34s} macro F1 = {scores.mean():.4f} ± {scores.std():.4f}   ({time.time()-t0:.0f}초)")
    return scores, np.array(w_log)


BASE_MODEL = lgb_tuned if "BEST" in dir() else lgb_base     # 실험 B를 돌렸으면 그 설정 사용
s, W = evaluate_with_weight_tuning(BASE_MODEL, tag="⑤ 클래스별 판정 기준 조정")
RESULTS.append(compare(s, base_scores, "⑤ 클래스별 판정 기준 조정"))

print("\n폴드별로 학습된 배율의 평균 (1보다 크면 '더 쉽게 선택되도록' 조정된 것):")
print(pd.DataFrame({"클래스": CLASSES, "평균 배율": W.mean(0).round(2),
                    "표준편차": W.std(0).round(2),
                    "샘플 수": np.bincount(y)}).to_string(index=False))

## 실험 D. 스태킹 앙상블

**아이디어.** 지금의 소프트 보팅은 세 모델의 확률을 **똑같은 비중으로** 평균한다.
하지만 "K_Scratch는 XGBoost가 잘 맞히고 Dirtiness는 랜덤포레스트가 낫다"처럼
모델마다 잘하는 클래스가 다르다면, 비중을 고정하는 건 손해다.

스태킹은 각 모델의 확률을 입력으로 받는 **메타 모델**을 하나 더 얹어
"어느 모델 말을 언제 믿을지"를 데이터로 학습한다.

In [ ]:
# ── 6. 실험 D: 스태킹 ────────────────────────────────────────────
from sklearn.ensemble import StackingClassifier

def make_stack():
    lgb = LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=31, subsample=0.8,
                         colsample_bytree=0.8, class_weight="balanced",
                         random_state=SEED, n_jobs=-1, verbose=-1)
    xgb = XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=5, subsample=0.8,
                        colsample_bytree=0.8, tree_method="hist", eval_metric="mlogloss",
                        random_state=SEED, n_jobs=-1)
    rf  = RandomForestClassifier(n_estimators=400, class_weight="balanced_subsample",
                                 random_state=SEED, n_jobs=-1)
    et  = ExtraTreesClassifier(n_estimators=400, class_weight="balanced",
                               random_state=SEED, n_jobs=-1)
    return StackingClassifier(
        estimators=[("lgb", lgb), ("xgb", xgb), ("rf", rf), ("et", et)],
        final_estimator=LogisticRegression(max_iter=2000, C=1.0, class_weight="balanced"),
        stack_method="predict_proba",
        cv=StratifiedKFold(5, shuffle=True, random_state=SEED),   # 내부 OOF로 메타 학습
        n_jobs=1, passthrough=False)

s = evaluate(make_stack, tag="⑥ 스태킹 앙상블")
RESULTS.append(compare(s, base_scores, "⑥ 스태킹 앙상블"))

## 실험 E. 불균형 데이터 처리 (리샘플링)

보고서 7.3절에서 요구하는 항목이다. **핵심 주의사항:** 리샘플링은 반드시 **학습 폴드에만**
적용해야 한다. 전체 데이터에 SMOTE를 먼저 돌리고 나누면, 평가 데이터와 거의 같은 합성 샘플이
학습에 들어가 성능이 크게 부풀려진다. 이걸 막으려고 `imblearn`의 Pipeline을 쓴다
(sklearn의 Pipeline과 달리 **학습할 때만** 리샘플링을 적용한다).

In [ ]:
# ── 7. 실험 E: 리샘플링 비교 ─────────────────────────────────────
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE, BorderlineSMOTE
from imblearn.combine import SMOTETomek

k = 4   # Dirtiness가 55개 → 학습 폴드에선 44개 남으므로 이웃 수를 작게

samplers = {
    "⑦ SMOTE":           SMOTE(k_neighbors=k, random_state=SEED),
    "⑧ BorderlineSMOTE": BorderlineSMOTE(k_neighbors=k, random_state=SEED),
    "⑨ SMOTE+Tomek":     SMOTETomek(smote=SMOTE(k_neighbors=k, random_state=SEED),
                                    random_state=SEED),
}
for name, samp in samplers.items():
    s = evaluate(lambda sp=samp: ImbPipeline([("sampler", sp), ("model", lgb_base())]), tag=name)
    RESULTS.append(compare(s, base_scores, name))

## 최종 조합과 결과 정리

가장 효과가 컸던 요소들을 합쳐 최종 모델을 만들고, 기준선과 정식으로 비교한다.
`evaluate_with_weight_tuning`에 스태킹을 넣으면 **D + C**를 동시에 적용한 것이 된다.

In [ ]:
# ── 8. 최종 조합 ─────────────────────────────────────────────────
final_scores, W_final = evaluate_with_weight_tuning(
    make_stack, n_inner=4, tag="⑩ 스태킹 + 판정 기준 조정")
RESULTS.append(compare(final_scores, base_scores, "⑩ 스태킹 + 판정 기준 조정"))

tbl = pd.DataFrame(RESULTS).round(4)
display(tbl)
tbl.to_csv("results_macro_f1_experiments.csv", index=False, encoding="utf-8-sig")

In [ ]:
# ── 9. 성능 비교 그래프 (보고서 figures/model_comparison.png) ────
fig, ax = plt.subplots(figsize=(9, 4.5))
t = tbl.sort_values("macro_f1")
ax.barh(t["실험"], t["macro_f1"], xerr=t["표준편차"], color="#4C78A8", alpha=.85, capsize=3)
ax.axvline(base_scores.mean(), color="crimson", ls="--", lw=1.2, label="기준선")
ax.set_xlim(0.75, max(0.88, t["macro_f1"].max() + .02))
ax.set_xlabel("Macro F1 (5-fold × 3회 반복 평균)")
ax.legend(); ax.grid(axis="x", alpha=.3)
plt.tight_layout(); plt.savefig("figures/model_comparison.png", dpi=150); plt.show()

In [ ]:
# ── 10. 최종 모델의 클래스별 성적 & 혼동행렬 ─────────────────────
# 단일 분할에서 상세 리포트 (보고서 8.2절용)
tr, te = next(iter(StratifiedKFold(5, shuffle=True, random_state=SEED).split(X, y)))

inner = StratifiedKFold(4, shuffle=True, random_state=SEED)
oof = np.zeros((len(tr), len(CLASSES)))
for itr, ite in inner.split(X.iloc[tr], y[tr]):
    mi = make_stack(); mi.fit(X.iloc[tr].iloc[itr], y[tr][itr])
    oof[ite] = mi.predict_proba(X.iloc[tr].iloc[ite])
w_best, _ = tune_weights(oof, y[tr])

m = make_stack(); m.fit(X.iloc[tr], y[tr])
pred = (m.predict_proba(X.iloc[te]) * w_best).argmax(1)

print(classification_report(y[te], pred, target_names=CLASSES, digits=3))

cm = confusion_matrix(y[te], pred, normalize="true")
fig, ax = plt.subplots(figsize=(6.5, 5.5))
im = ax.imshow(cm, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(len(CLASSES)), CLASSES, rotation=45, ha="right")
ax.set_yticks(range(len(CLASSES)), CLASSES)
ax.set_xlabel("예측"); ax.set_ylabel("실제")
for i in range(len(CLASSES)):
    for j in range(len(CLASSES)):
        if cm[i, j] > .005:
            ax.text(j, i, f"{cm[i,j]:.2f}", ha="center", va="center",
                    color="white" if cm[i, j] > .5 else "black", fontsize=9)
fig.colorbar(im, shrink=.8)
plt.tight_layout(); plt.savefig("figures/confusion_matrix.png", dpi=150); plt.show()

## 과적합 점검 — 이 과정을 꼭 거칠 것

위 실험들은 **같은 15개 폴드를 수십 번 들여다보며** 좋은 방법을 고른 것이다.
그 과정 자체가 폴드에 조금씩 맞춰지는 행위라, 최종 숫자는 실제보다 낙관적일 수 있다.
**발표에서 가장 날카로운 질문이 여기서 나온다.**

방어책은 간단하다. 실험에 **한 번도 쓰지 않은** 분할(다른 난수)에서 기준선과 최종 모델을
한 번만 비교해, 개선폭이 그대로 재현되는지 확인한다. 재현되면 그 숫자를 보고서에 쓴다.

In [ ]:
# ── 11. 다른 난수 분할에서 재현성 확인 ───────────────────────────
HOLD = list(RepeatedStratifiedKFold(n_splits=5, n_repeats=2, random_state=20261014).split(X, y))

b2 = evaluate(lgb_base,  folds=HOLD, tag="기준선 (새 분할)")
f2, _ = evaluate_with_weight_tuning(make_stack, tag="최종 모델 (새 분할)")  # FOLDS 사용 주의
# ↑ 재현성 확인을 원하면 evaluate_with_weight_tuning 안의 FOLDS를 HOLD로 바꿔 한 번 더 실행

print(f"\n새 분할에서의 개선폭: {f2.mean() - b2.mean():+.4f}")
print("원래 분할에서의 개선폭:", f"{final_scores.mean() - base_scores.mean():+.4f}")
print("두 값이 비슷하면 개선은 진짜, 크게 줄어들면 폴드에 과적합된 것")

---

## 돌려본 뒤 해석하는 법

| 상황 | 의미 | 다음 행동 |
|---|---|---|
| ⑤(판정 기준 조정)에서 크게 오름 | 모델은 이미 충분히 학습했고, **결정 규칙만 어긋나 있었다** | 이걸 최종안으로. 배율표를 보고서에 넣고 "왜 희귀 클래스에 큰 배율이 붙었는지" 해석 |
| ②③(클래스 가중치)에서 오름 | 학습 목표와 평가 지표의 불일치가 원인이었다 | ⑤와 중복될 수 있으니 둘 다 쓰면 과해지는지 확인 |
| ④(튜닝)만 오름 | 기존 설정이 보수적이었을 뿐 | 탐색 횟수를 늘려 조금 더 짜낸다 |
| ⑦⑧⑨(리샘플링)가 기준선보다 낮음 | **정상이다.** 트리 모델에서 흔한 결과 | "리샘플링보다 가중치·임계값 조정이 유효했다"는 결론으로 7.3절을 쓴다 |
| 전부 0.83 언저리에서 멈춤 | 변수가 가진 정보의 한계 | Other_Faults를 분리한 2단계 모델, 또는 해당 클래스 제외 후 성능 보고 |

**마지막으로 현실적인 이야기.** 이 데이터(UCI Steel Plates Faults)에서 7클래스 macro F1은
보통 0.78~0.84 구간에 모인다. 0.84는 닿을 수 있는 범위지만 여유 있는 목표는 아니다.
숫자가 0.83에서 멈추더라도, **"무엇을 시도했고 왜 안 되었는지를 검정으로 확인했다"** 는 기록이
0.84라는 숫자보다 발표에서 강하다. 실패한 실험도 전부 표에 남겨두는 게 좋다.
